# Stage 1: create training data from OSZ files

Beat! Maps! They're all in the file!

If you wanna run models, I'm sure you'll find

The steps for training, aren't that overwhelming,

Nevertheless you'll get a lesson on data conversion now

In [1]:
import sys
sys.path.append("..")  # add parent directory to path
import os
import pandas as pd

from beatlearning import OsuBeatConverter


# loads all ".osz" files from "./songs" and its subfolders, saves converted results to "./output"
osudata = OsuBeatConverter("./songs", "./output")

In [2]:
for i, file in enumerate(osudata):
    print(f"Converting '{file}' ({i + 1} / {len(osudata)})")
    
    extract = osudata.extract(file)
    """
    extract all audio files, beatmaps and their metadata from the current .osz file and cache them:
    
    extract = {
        "beatmaps": {
            "readable_beatmap_file_name_1" : {
                "meta": {...},
                "data": pd.DataFrame
            },
            ...
        },
        "audio": {
            "readable_audio_file_name_for_beatmaps_1": [encodec_tokens_list_1, encodec_tokens_list_2],
            ...
        }
    }
    """;
    
    # go through all found beatmaps
    results = None
    for beatmap in extract["beatmaps"]:
        sel = extract["beatmaps"][beatmap]
        audio = extract["audio"][sel["meta"]["audio"]]
        
        # only include some of the metadata in the final dataset
        meta = {
            "id": sel["meta"]["info"]["BeatmapID"] if "info" in sel["meta"] and "BeatmapID" in sel["meta"]["info"] else 0,
            "bid": sel["meta"]["info"]["BeatmapSetID"] if "info" in sel["meta"] and "BeatmapSetID" in sel["meta"]["info"] else 0,
            "mode": sel["meta"]["mode"],  # 0 - Osu, 1 - Taiko, 2 - Catch, 3 - Mania
            "difficulty": sel["meta"]["level"] / 10.,  # cast to 0. - 1. float (passed to model)
        }
        
        # convert the cached intermediate beatmap representation into training data
        result = osudata.convert(sel["data"], audio, meta)
        if result is not None:
            results = pd.concat([results, result], ignore_index=True)  # add it to the rest of the data
    
    if results is not None:
        # save the combined data to the output folder as a .beat (parquet) file with the same name
        output_file = os.path.join(osudata.output_folder, f"{os.path.splitext(os.path.basename(file))[0]}.beat")
        print(f"Saving '{output_file}'")
        results.to_parquet(output_file, engine="pyarrow", index=False)
        
print("Done.")

Converting './songs\18260 Masayoshi Minoshima feat. nomico - Bad Apple!! [no video].osz' (1 / 1)


100%|████████████████████████████████████████████████████████████████████████████████| 220/220 [03:29<00:00,  1.05it/s]


Saving './output\18260 Masayoshi Minoshima feat. nomico - Bad Apple!! [no video].beat'
Done.


In [3]:
# contents of the merged .beat file
pd.read_parquet("./output/18260 Masayoshi Minoshima feat. nomico - Bad Apple!! [no video].beat")

,audio_tokens_1,audio_tokens_2,tempo,offset,time,weight,hits_l,hits_l_token_output,hits_l_token_inputs,hits_d,...,holds_r,holds_r_token_output,holds_r_token_inputs,holds_a,holds_a_token_output,holds_a_token_inputs,id,bid,mode,difficulty
0,"[62, 62, 62, 62, 408, 408, 62, 408, 62, 408, 6...","[913, 424, 424, 424, 544, 544, 424, 913, 424, ...",0.434783,0.333000,1.0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0.2
1,"[75, 38, 38, 170, 142, 170, 170, 38, 511, 511,...","[867, 755, 240, 883, 660, 733, 210, 637, 735, ...",0.434783,0.333000,1.0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0.2
2,"[2, 45, 525, 201, 511, 927, 221, 927, 221, 927...","[158, 914, 209, 914, 57, 100, 29, 57, 698, 57,...",0.434783,0.072130,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0.2
3,"[46, 38, 161, 964, 101, 968, 101, 827, 140, 94...","[973, 181, 867, 698, 210, 411, 210, 411, 57, 9...",0.434783,0.376217,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0.2
4,"[675, 75, 617, 946, 75, 675, 380, 140, 545, 40...","[491, 909, 867, 892, 57, 57, 411, 168, 57, 152...",0.434783,0.376217,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1072,"[358, 747, 324, 385, 734, 747, 203, 925, 582, ...","[524, 345, 624, 626, 872, 350, 259, 85, 220, 9...",0.434783,0.107478,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1, 9, 1, 1, 1, 17, 1, 5, 1, 1, 1, 513, 1, 129...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]",1058,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0.5
1073,"[747, 747, 925, 747, 984, 925, 747, 850, 696, ...","[69, 899, 634, 174, 300, 616, 948, 220, 349, 9...",0.434783,0.411826,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[9, 1, 1, 1, 17, 1, 5, 1, 1, 1, 513, 1, 129, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]",131,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0.5
1074,"[747, 203, 203, 747, 253, 747, 43, 613, 358, 4...","[375, 198, 178, 528, 712, 289, 714, 520, 375, ...",0.434783,0.281391,1.0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1, 1, 1, 17, 1, 5, 1, 1, 1, 513, 1, 129, 10, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0]",265,"[1, 1, 1, 1, 1, 1, 1, 1,

That's it for today

Good job! You can move on to the next stage now!